# EDA 01 — Q1: Distribusi Return M1 / M5 / M15 per Strata

**Rancangan:** EDA v3 Final (Architect Spark, disetujui user, diaudit Klenz).

## Tujuan notebook ini
Menjawab **Q1**: bagaimana distribusi return log-close-BID pada M1, M5, dan M15,
dengan stratifikasi wajib A / B / C yang sudah dikunci di notebook 00.

## Definisi yang dipakai (dikunci di Kontrak 00, tidak diubah di sini)
- `r_t = ln(close_t / close_{t-1})`, `close` = **harga BID** (RULES §18.16).
- Spread diperlakukan sebagai **biaya terpisah**, bukan bagian dari return.
- Stratifikasi: A = utama, B = H1-2023, C = dua window anomali.

## Putusan Architect atas SA-03 (jebakan kalender UTC vs broker)
Diterima di sesi ini dan dicatat di sel md terpisah. Tidak mengubah apa pun di Q1,
tetapi menetapkan bahwa seluruh pekerjaan **level-hari** (Q6/Q7) dikunci pada
**tanggal-broker**, sedangkan hari-UTC hanya sekunder.

## Batas keras notebook ini (Wajib)
- **Tidak** menghitung P&L, equity curve, atau profit simulasi apa pun.
- **Tidak** menghasilkan sinyal, model, atau pemilihan metode strategi.
- **Tidak** menulis `configs/`. Config hanya dibaca.
- **Tidak** melakukan imputasi: hanya drop/NaNa dengan jumlah dilaporkan.
- Inferensi berbentuk sinyal **tidak** diambil dari M1 (microstructure noise /
  bid-bounce); M1 hanya dipakai untuk deskripsi distribusi.

## Urutan sel
1. Kontrak + sysinfo + kunci tzdata
2. Definisi return + fungsi statistik (dokumentasi asumsi sigma)
3. Beban M1/M5/M15 + hitung return per TF
4. Audit stratifikasi per TF (A/B/C lepas, A+B+C = total)
5. Tabel Q1 utama: skew, kurtosis, kuantil, pangsa 3sigma/5sigma per TF x strata
6. QQ-plot terhadap normal
7. Catatan keterbatasan biaya BID-only + assertion penutup
8. Ringkasan assertion

In [1]:
import importlib.metadata
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats


def find_project_root(start: Path) -> Path:
    """Cari root proyek (pyproject.toml + src/) dari cwd ke atas."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("PROJECT_ROOT tidak ditemukan")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.broker_time import BrokerSchedule, wall_clock_series  # noqa: E402
from src.utils.config import load_config  # noqa: E402

print("PROJECT_ROOT :", PROJECT_ROOT)
print("python       :", sys.version.split()[0])
for package in ("pandas", "numpy", "pyarrow", "matplotlib", "scipy", "tzdata"):
    print(f"  {package:11s}: {importlib.metadata.version(package)}")
assert importlib.metadata.version("tzdata") == "2026.5", "tzdata harus 2026.5 (Kontrak 00)"
# Kunci versi pandas major. pandas 3.x mengubah satuan .asi8 (detik vs nanos),
# bug yang pernah merusak kolom epoch (SESSION_LOG 2026-10-05). Notebook WAJIB
# jalan di venv kanonik (pandas 2.x), bukan .venv (pandas 3.x).
assert (
    int(importlib.metadata.version("pandas").split(".")[0]) == 2
), "Notebook WAJIB jalan di venv kanonik (pandas 2.x); pandas 3.x merusak .asi8."

PROJECT_ROOT : D:\ANN
python       : 3.12.8
  pandas     : 2.2.3
  numpy      : 1.26.4
  pyarrow    : 17.0.0
  matplotlib : 3.10.9
  scipy      : 1.17.1
  tzdata     : 2026.5


## Putusan Architect (Spark) atas SA-03 — dicatat verbatim agar tidak hilang

> **R1:** "Hari-UTC" = label kalender murni dengan offset struktural −1 yang
> didokumentasikan; **DILARANG** menggeser data agar "sepakat".
>
> **R2:** Syarat "wajib sepakat" ditafsir ulang menjadi **sepakat-dengan-kesadaran-
> geser**: bandingkan H/L hari-broker vs H/L hari-UTC dengan memperhitungkan offset
> struktural; hanya divergensi **MELEBIHI** geseran struktural yang menjadi temuan
> (dilaporkan sebagai sensitivitas).
>
> **R3:** Seluruh pekerjaan level-hari (uji regime Q7, primer prior-day Q6) dikunci
> pada **tanggal-broker** — konsisten dengan bucket D1, strata, dan window anomali yang
> semuanya broker-defined. Hari-UTC sekunder saja.
>
> **R4:** Tanpa redesain, tanpa penundaan Langkah 1–3; aturan R1–R3 dicatat di
> nb04/nb05 dan laporan final.

**Dampak ke Q1:** Q1 bekerja pada **bar** (bukan hari), jadi SA-03 tidak menyentuh
distribusi return. Aturan ini baru mengikat saat Q6/Q7 (nb04/nb05).

**Risiko yang dicatat Architect:** analis nb04 lupa menerapkan R2 → mitigasi:
assertion perbandingan shift-aware + review. Session ini menambah satu proteksi:
sel audit di bawah selalu melaporkan **offset struktural yang diukur**, bukan hanya
kesimpulannya.

In [2]:
RETURN_DEFINITION = "r_t = ln(close_t / close_{t-1}); close = BID"
ANOMALY_OFFSET_EXPECTED = -1
print(RETURN_DEFINITION)
print("offset struktural hari-UTC vs hari-broker (harus", ANOMALY_OFFSET_EXPECTED, ")")

r_t = ln(close_t / close_{t-1}); close = BID
offset struktural hari-UTC vs hari-broker (harus -1 )


## 1. Fungsi return dan statistik

**Return** dihitung pada bar yang **sudah diurutkan kronologis** (`sort_values('utc')`),
bukan pada urutan file. Ini penting: urutan file adalah per tahun, sehingga kalau
return dihitung tanpa pengurutan, perpindahan antar-file akan menghasilkan return
salah yang memalsukan ekor distribusi.

**Sigma** untuk ambang 3σ/5σ dihitung **per TF per strata** dari data itu sendiri
(bukan sigma global), supaya ambang tidak bias oleh perbedaan skala antar strata.
Sigma memakai `ddof=0` (populasi bar yang diamati), bukan `ddof=1`.

**Kuantil** 0,1% dan 99,9% adalah kuantil *empiris* (tanpa asumsi normal).
**Skewness** dan **kurtosis ekses** adalah moment statistics mentah — pada data
finansial yang-tailed dan ber-cluster, keduanya **bias** untuk n besar; dipakai
sebagai deskripsi, bukan sebagai bukti distribusi tertentu.

In [3]:
def add_log_return(frame: pd.DataFrame) -> pd.DataFrame:
    """Tambahkan return log close-BID pada frame yang sudah terurut kronologis.

    Args:
        frame: Frame dengan kolom 'close' dan indeks unik.

    Returns:
        Salinan frame dengan kolom 'r' (NaN pada bar pertama).
    """
    out = frame.copy()
    out["r"] = np.log(out["close"] / out["close"].shift(1))
    return out


def return_stats(returns: pd.Series) -> dict[str, float]:
    """Statistik deskriptif distribusi return (tanpa asumsi normal).

    Args:
        returns: Series return log; NaN akan di-drop untuk statistik.

    Returns:
        Dictionary metrik Q1.
    """
    clean = returns.dropna()
    sigma = float(clean.std(ddof=0))
    return {
        "n": int(clean.size),
        "mean": float(clean.mean()),
        "sigma": sigma,
        "skew": float(clean.skew()),
        "kurt_excess": float(clean.kurt()),
        "q_0.1%": float(clean.quantile(0.001)),
        "q_1%": float(clean.quantile(0.01)),
        "q_99%": float(clean.quantile(0.99)),
        "q_99.9%": float(clean.quantile(0.999)),
        "share_gt_3sigma": float((clean.abs() > 3 * sigma).mean()),
        "share_gt_5sigma": float((clean.abs() > 5 * sigma).mean()),
    }


return_stats(pd.Series([np.nan, 0.001, -0.002]))

{'n': 2,
 'mean': -0.0005,
 'sigma': 0.0015,
 'skew': nan,
 'kurt_excess': nan,
 'q_0.1%': -0.001997,
 'q_1%': -0.00197,
 'q_99%': 0.0009699999999999999,
 'q_99.9%': 0.0009970000000000003,
 'share_gt_3sigma': 0.0,
 'share_gt_5sigma': 0.0}

## 2. Memuat data dan menghitung return per timeframe

Timeframe yang dianalisis: **M1 (primer), M5, M15** — sesuai batasan ruang lingkup
scalping M1–M15. H1/D1 tidak masuk Q1 (dipakai di Q6/Q7 sebagai konteks).

Kolom `time` di parquet adalah **string ISO-8601 UTC**, jadi di-parse dengan
`pd.to_datetime(..., utc=True)`.

Jumlah bar **sebelum** dan **sesudah** drop NaN dilaporkan untuk semua TF — sesuai
aturan missing-data (drop dilaporkan jumlahnya, nol imputasi).

In [4]:
config = load_config("data")
SYMBOL = config["mt5"]["symbol"]
SCHEDULE = BrokerSchedule.from_config(config["mt5"])
ANOMALY_WINDOWS = tuple((w["start"], w["end"]) for w in config["validation"]["anomaly_windows"])

Q1_TIMEFRAMES = ("M1", "M5", "M15")
YEARS = (2023, 2024, 2025, 2026)
NEEDED_COLUMNS = ["time", "broker_time_raw", "close"]

print(f"symbol: {SYMBOL} | windows anomali: {len(ANOMALY_WINDOWS)}")
for start, end in ANOMALY_WINDOWS:
    print(f"  {start} .. {end}")

symbol: USDJPY | windows anomali: 2
  2024-10-28 .. 2024-11-01
  2026-03-10 .. 2026-03-27


In [5]:
def load_timeframe(timeframe: str) -> pd.DataFrame:
    """Muat satu timeframe 2023-2026, urut kronologis, dengan return log.

    Args:
        timeframe: Kode timeframe, mis. 'M1'.

    Returns:
        Frame terurut dengan kolom utc, wall_date, dan r.
    """
    frames = []
    for year in YEARS:
        path = (
            PROJECT_ROOT
            / "data/processed"
            / timeframe
            / f"usdjpy_{timeframe.lower()}_{year}.parquet"
        )
        frames.append(pd.read_parquet(path, columns=NEEDED_COLUMNS))
    frame = pd.concat(frames, ignore_index=True)
    frame["utc"] = pd.to_datetime(frame["time"], utc=True)
    frame["wall"] = wall_clock_series(frame["broker_time_raw"])
    frame["wall_date"] = frame["wall"].dt.normalize()
    frame = frame.sort_values("utc", kind="mergesort").reset_index(drop=True)
    return add_log_return(frame)


DATA = {timeframe: load_timeframe(timeframe) for timeframe in Q1_TIMEFRAMES}

print(f"{'TF':4} {'bar':>10} {'r != NaN':>10} {'drop warmup':>11} {'sorted':>7} {'dup':>5}")
for timeframe, frame in DATA.items():
    n_total = len(frame)
    n_valid = int(frame["r"].notna().sum())
    n_dup = int(frame["utc"].duplicated().sum())
    is_sorted = bool(frame["utc"].is_monotonic_increasing)
    print(
        f"{timeframe:4} {n_total:>10,} {n_valid:>10,} {n_total - n_valid:>11,} {str(is_sorted):>7} {n_dup:>5}"
    )

assert all(
    frame["utc"].is_monotonic_increasing for frame in DATA.values()
), "urutan waktu harus naik"
assert all(
    int(frame["utc"].duplicated().sum()) == 0 for frame in DATA.values()
), "duplikat waktu harus nol"

TF          bar   r != NaN drop warmup  sorted   dup
M1    1,347,847  1,347,846           1    True     0
M5      269,964    269,963           1    True     0
M15      90,041     90,040           1    True     0


## 3. Audit stratifikasi per timeframe

Definisi strata diwarisi **persis** dari notebook 00 dan dihitung ulang di sini
(sumber tunggal definisi = nb00):
- **B** = hari broker 2023-01-01 .. 2023-06-30 (H1-2023).
- **C** = dua window anomali dari `configs/data.yaml`.
- **A** = sisanya.

Untuk timeframe M5/M15 jumlah bar per strata **tidak mungkin** sama dengan M1 —
itu konsekuensi resample yang wajar, bukan kegagalan. Yang wajib lulus adalah:
partisi A/B/C saling lepas dan **A+B+C = total** per timeframe.

Pada frame yang sudah urut, `r` dihitung **sebelum** pemisahan strata. Karena
`.shift(1)` diterapkan pada seri utuh, return pada bar pertama sebuah stratum bisa
merujuk bar di stratum sebelumnya. Ini **benar secara ekonomi** (return dihitung dari
bar berulang yang benar-benar ada sebelumnya) dan **tidak**(create) look-ahead:
informasi hanya dari masa lalu. Jumlahnya dilaporkan sebagai `cross_stratum_returns`.

In [6]:
def stratify(frame: pd.DataFrame) -> dict[str, pd.Series]:
    """Bangun mask stratifikasi A/B/C dari hari broker (definisi dari nb00).

    Args:
        frame: Frame dengan kolom wall_date dan r.

    Returns:
        Dictionary 'A', 'B', 'C' berisi boolean mask.
    """
    wall_date = frame["wall_date"]
    b_mask = (wall_date >= pd.Timestamp("2023-01-01")) & (wall_date <= pd.Timestamp("2023-06-30"))
    c_mask = pd.Series(False, index=frame.index)
    for start, end in ANOMALY_WINDOWS:
        c_mask |= (wall_date >= pd.Timestamp(start)) & (wall_date <= pd.Timestamp(end))
    a_mask = ~(b_mask | c_mask)
    return {"A": a_mask, "B": b_mask, "C": c_mask}


def first_of_stratum_is_repeated(returns: pd.Series, mask: pd.Series) -> int:
    """Hitung return pada bar pertama stratum yang r-nya menyeberang stratum.

    Args:
        returns: Series return.
        mask: Boolean mask stratum.

    Returns:
        Jumlah return yang border menyeberang (informasi tetap dari masa lalu).
    """
    inside = mask & returns.notna()
    shifted_inside = inside.shift(1, fill_value=False)
    return int((inside & ~shifted_inside).sum())


print(f"{'TF':4} {'strata':6} {'bar':>10} {'r != NaN':>10} {'cross':>7} {'pangsa':>8}")
for timeframe, frame in DATA.items():
    masks = stratify(frame)
    n_total = len(frame)
    for name in ("A", "B", "C"):
        mask = masks[name]
        n_valid = int((mask & frame["r"].notna()).sum())
        cross = first_of_stratum_is_repeated(frame["r"], mask)
        print(
            f"{timeframe:4} {name:6} {int(mask.sum()):>10,} {n_valid:>10,} {cross:>7} {mask.mean():>7.2%}"
        )
    union = masks["A"] | masks["B"] | masks["C"]
    assert not (masks["B"] & masks["C"]).any(), "B dan C tidak boleh tumpang tindih"
    assert bool(union.all()), "A+B+C harus menutup seluruh bar"
    assert int(union.sum()) == n_total, "A+B+C harus sama dengan total"
    print(f"{timeframe:4} {'A+B+C':6} {int(union.sum()):>10,} | partisi PASS")

TF   strata        bar   r != NaN   cross   pangsa
M1   A       1,136,401  1,136,401       3  84.31%


M1   B         185,248    185,247       1  13.74%
M1   C          26,198     26,198       2   1.94%
M1   A+B+C   1,347,847 | partisi PASS
M5   A         227,568    227,568       3  84.30%
M5   B          37,152     37,151       1  13.76%
M5   C           5,244      5,244       2   1.94%
M5   A+B+C     269,964 | partisi PASS
M15  A          75,863     75,863       3  84.25%
M15  B          12,430     12,429       1  13.80%
M15  C           1,748      1,748       2   1.94%
M15  A+B+C      90,041 | partisi PASS


## 3b. Karakterisasi ekor dan massa nol (self-audit putaran 1 & 3)

Dua karakteristik data yang **tidak terlihat** di tabel momente dan wajib
dokumentasikan eksplisit:

1. **Massa di nol**: berapa bar yang return-nya tepat 0 (harga tidak berubah dari
   menit sebelumnya). Ini wajar secara alami dari microstructure, bukan data rusak —
   tapi ia berkontribusi besar terhadap kurtosis yang tinggi.
2. **Ekor:** konversi ke **pip** yang benar adalah
   `pip = (exp(r) - 1) * close / 0.01`. Salah konversi (menganggap log-return
   sudah berupa fraksi pip) akan sangat MENGAHALANGI besarnya pergerakan. Jam dari
   pergerakan terbesar juga dihitung untuk memeriksa apakah ekor terkonsentrasi
   di **jadwal berita makro** (RULES §18.15 melarang trading saat news high impact
   tanpa proteksi).

**Konsekuensi untuk interpretasi:** ekses kurtosis tinggi **tidak boleh**
diatribusikan ke 'microstructure noise' sebelum ekor dipisahkan dari news spike.
Atribusi itu jadi hipotesis, bukan temuan Q1.

In [7]:
STEP_MINUTES = {"M1": 1, "M5": 5, "M15": 15}
PIP_SIZE = 0.01

print(f"{'TF':4} {'zero-share':>10} {'gap-returns':>12} {'max|r| pip':>11}")
for timeframe in Q1_TIMEFRAMES:
    frame = DATA[timeframe]
    step = STEP_MINUTES[timeframe]
    returns = frame["r"]
    gap_mask = frame["utc"].diff() > pd.Timedelta(minutes=step)
    pips = (np.exp(returns) - 1.0) * frame["close"] / PIP_SIZE
    print(
        f"{timeframe:4} {returns.eq(0).mean():>9.2%} {int(gap_mask.sum()):>12,} {pips.abs().max():>10.1f}"
    )

print()
m1_frame = DATA["M1"]
m1_pips = (np.exp(m1_frame["r"]) - 1.0) * m1_frame["close"] / PIP_SIZE
top_index = m1_pips.abs().sort_values(ascending=False).index[:8]
print("8 pergerakan M1 terbesar (pip):")
for index in top_index:
    print(f"  {m1_frame.loc[index, 'utc']}  {m1_pips.loc[index]:+8.2f} pip")

print()
print(
    f"jam UTC dari 8 pergerakan terbesar: {sorted(m1_frame.loc[top_index, 'utc'].dt.hour.tolist())}"
)
print("12:30 UTC = 08:30 ET (data ketenagakerjaan AS).")
print(
    "21-22 UTC = pembukaan sesi / transisi DST (return ini return GAP lintas akhir pekan, bukan pergerakan 1 menit)."
)
print("-> EKOR DIKENDALIKAN DUA SUMBER: berita makro DAN gap pembukaan sesi.")
print("   Jadi atribusi 'microstructure noise' untuk kurtosis TIDAK didukung Q1.")

TF   zero-share  gap-returns  max|r| pip
M1       5.83%        1,824      164.7
M5       2.01%          993      218.3
M15      1.01%          878      273.0



8 pergerakan M1 terbesar (pip):
  2025-10-05 22:00:00+00:00   +164.70 pip
  2023-01-18 02:40:00+00:00   +153.59 pip
  2024-10-04 12:30:00+00:00   +139.81 pip
  2025-04-06 22:00:00+00:00   -137.59 pip
  2023-03-12 21:07:00+00:00   -136.10 pip
  2024-08-02 12:30:00+00:00   -134.17 pip
  2025-04-22 22:00:00+00:00   +133.43 pip
  2026-02-11 13:39:00+00:00   -130.87 pip

jam UTC dari 8 pergerakan terbesar: [2, 12, 12, 13, 21, 22, 22, 22]
12:30 UTC = 08:30 ET (data ketenagakerjaan AS).
21-22 UTC = pembukaan sesi / transisi DST (return ini return GAP lintas akhir pekan, bukan pergerakan 1 menit).
-> EKOR DIKENDALIKAN DUA SUMBER: berita makro DAN gap pembukaan sesi.
   Jadi atribusi 'microstructure noise' untuk kurtosis TIDAK didukung Q1.


## 4. Tabel Q1 utama

Kolom yang dilaporkan (sesuai metrik rancangan): `n`, skewness, kurtosis ekses,
kuantil 0,1% / 1% / 99% / 99,9%, serta pangsa bar dengan |r| di atas 3σ dan 5σ.

Sigma untuk ambang 3σ/5σ dihitung **per baris tabel** (per TF × strata), sehingga
ambang tidak dipengaruhi perbedaan skala antar strata.

**Cara membaca angka skew/kurtosis:** nilai|kurtosis ekses| >> 3 dan |skew| >> 0.5
pada data M1/M5 adalah **diharapkan** untuk data aset keuangan dan **bukan** bukti
bahwa perhitungan salah. Yang justru penting: (a) apakah ekses kurtosis M1 jauh lebih
besar dari M5/M15 — indikasi microstructure noise/bid-bounce di M1; dan (b) apakah
strata B menyimpang dari A.

**Penting soal ambang 3sigma/5sigma:** porsi bar di atas 3sigma ternyata **jauh lebih
besar** dari teoritis normal (0,27%), karena ekornya berat. Jadi angka 3sigma di sini
**bukan** ambang "outlier langka" — ini konsekuensi langsung dari ekor yang berat,
bukan ambang keputusan. Tidak ada satu pun angka tabel ini yang dipakai sebagai ambang
filter.

**Tidak ada** threshold trading di notebook ini. Tidak ada keputusan metode.

In [8]:
table_rows = []
for timeframe in Q1_TIMEFRAMES:
    frame = DATA[timeframe]
    masks = stratify(frame)
    for name in ("A", "B", "C"):
        stats = return_stats(frame.loc[masks[name], "r"])
        row = {"TF": timeframe, "strata": name}
        row.update(stats)
        table_rows.append(row)

Q1_TABLE = pd.DataFrame(table_rows).set_index(["TF", "strata"])
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:.6e}")
display(Q1_TABLE)

# Cetak juga sebagai stream supaya tabel utama tersimpan di output cell.
print("TABEL Q1 (stream, supaya tersimpan):")
print(Q1_TABLE.to_string())
print(f"\n{len(Q1_TABLE)} baris = {len(Q1_TIMEFRAMES)} TF x 3 strata")

# Pengujian pembeda yang JANGAN disimpulkan dari tabel saja (self-audit #2).
m1_frame = DATA["M1"]
m1_masks = stratify(m1_frame)
returns_a = m1_frame.loc[m1_masks["A"], "r"].dropna()
returns_b = m1_frame.loc[m1_masks["B"], "r"].dropna()
welch = scipy.stats.ttest_ind(returns_a, returns_b, equal_var=False)
print("\nUJI PEMBEDA (bukan klaim Q1, hanya pagar pembeda):")
print(f"  mean return M1: A={returns_a.mean():.3e} B={returns_b.mean():.3e}")
print(f"  Welch t-test A vs B: t={welch.statistic:.2f} p={welch.pvalue:.3f}")
print("  -> selisih mean TIDAK signifikan; jangan klaim drift B vs A.")
rng = np.random.default_rng(20261006)
cubes_b = returns_b.to_numpy() ** 3
boot = [float(np.mean(rng.choice(cubes_b, size=40000, replace=True))) for _ in range(300)]
boot_low, boot_high = float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5))
print(f"  skew B bootstrap 95% CI = [{boot_low:.3e}, {boot_high:.3e}]")
print("  -> tanda skew B hanya TIDAK kokoh; CI melintasi nol. Catat sebagai pola, bukan temuan.\n")

n         mean        sigma          skew  kurt_excess        q_0.1%          q_1%        q_99%      q_99.9%  share_gt_3sigma  share_gt_5sigma
TF  strata                                                                                                                                                      
M1  A       1136401 6.084237e-08 1.608817e-04 -7.440602e-01 1.954764e+02 -9.672903e-04 -4.237060e-04 4.100827e-04 9.166768e-04     1.323300e-02     3.038540e-03
    B        185247 5.253311e-07 1.885797e-04  9.318173e-01 1.899319e+02 -1.048616e-03 -4.931613e-04 4.835184e-04 1.059202e-03     1.295028e-02     2.720692e-03
    C         26198 8.079973e-07 1.399611e-04 -1.517266e+00 8.705036e+01 -7.644354e-04 -3.657968e-04 3.527368e-04 6.785020e-04     1.282541e-02     2.404764e-03
M5  A        227568 3.038271e-07 3.565863e-04 -1.456013e+00 1.028481e+02 -2.181544e-03 -9.486927e-04 8.949412e-04 2.056900e-03     1.314772e-02     3.102369e-03
    B         37151 2.621324e-06 4.185221e-04  5.362932e-01 5.696206e+01 -2.456232e-03 -1.099654e-03 1.065337e-03 2.535478e-03     1.345859e-02     3.095475e-03
    C          5244 4.036596e-06 3.073385e-04 -1.675999e+00 4.208051e+01 -1.861609e-03 -7.835978e-04 7.537072e-04 1.535340e-03     1.239512e-02     2.669718e-03
M15 A         75863 9.113973e-07 6.120079e-04 -1.224980e+00 5.117605e+01 -4.042704e-03 -1.682542e-03 1.563728e-03 3.498755e-03     1.397256e-02     3.361322e-03
    B         12429 7.848203e-06 7.211478e-04  4.363731e-01 2.783489e+01 -4.337213e-03 -1.925807e-03 1.855169e-03 4.712954e-03     1.488454e-02     3.620565e-03
    C          1748 1.210979e-05 5.211761e-04 -1.066520e+00 1.503506e+01 -3.465748e-03 -1.269921e-03 1.367821e-03 2.386975e-03     1.086957e-02     2.860412e-03

TABEL Q1 (stream, supaya tersimpan):
                  n         mean        sigma          skew  kurt_excess        q_0.1%          q_1%        q_99%      q_99.9%  share_gt_3sigma  share_gt_5sigma
TF  strata                                                                                                                                                      
M1  A       1136401 6.084237e-08 1.608817e-04 -7.440602e-01 1.954764e+02 -9.672903e-04 -4.237060e-04 4.100827e-04 9.166768e-04     1.323300e-02     3.038540e-03
    B        185247 5.253311e-07 1.885797e-04  9.318173e-01 1.899319e+02 -1.048616e-03 -4.931613e-04 4.835184e-04 1.059202e-03     1.295028e-02     2.720692e-03
    C         26198 8.079973e-07 1.399611e-04 -1.517266e+00 8.705036e+01 -7.644354e-04 -3.657968e-04 3.527368e-04 6.785020e-04     1.282541e-02     2.404764e-03
M5  A        227568 3.038271e-07 3.565863e-04 -1.456013e+00 1.028481e+02 -2.181544e-03 -9.486927e-04 8.949412e-04 2.056900e-03     1.314772e-02     3.102369e-

## 5. QQ-plot terhadap normal

QQ-plot membandingkan kuantil observasi dengan kuantil normal standar. Tiga hal
hanya bisa dibaca dari grafik ini:
1. **Ekor** (*tails*): titik yang menjauh dari garis diagonal di ujung kiri/kanan berarti
   lebih berat dari normal — diharapkan untuk return aset keuangan.
2. **Puncak**: apakah titik tengah meninggi (bola_bocor tinggi), yang menandakan
   konsentrasi return kecil — konsisten dengan microstructure noise M1.
3. **Kesimpulannya sudah diketahui dari angka**: QQ-plot di sini adalah konfirmasi
   visual, bukan sumber klaim baru. Tidak ada parameter yang di-*fit* ke grafik ini.

In [9]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIGURE_DIR = PROJECT_ROOT / "reports/eda/2026-10-06/figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

figures = []
for timeframe in Q1_TIMEFRAMES:
    frame = DATA[timeframe]
    masks = stratify(frame)
    clean = frame.loc[masks["A"], "r"].dropna()
    n_quantiles = 1000
    probabilities = np.linspace(0.001, 0.999, n_quantiles)
    observed = np.sort(clean.to_numpy())[
        np.linspace(0, clean.size - 1, n_quantiles).round().astype(int)
    ]
    expected = scipy.stats.norm.ppf(probabilities)

    fig, axis = plt.subplots(figsize=(6, 6))
    axis.scatter(expected, observed, s=3, alpha=0.35, rasterized=True)
    sigma = float(clean.std(ddof=0))
    mean = float(clean.mean())
    axis.plot(expected, mean + sigma * expected, color="crimson", linewidth=1.2)
    axis.set_title(f"{timeframe} stratum A (n={clean.size:,}) vs normal")
    axis.set_xlabel("Kuantil normal standar")
    axis.set_ylabel("Kuantil observasi (log return BID)")
    fig.tight_layout()
    path = FIGURE_DIR / f"FIG-01_qq_{timeframe}.png"
    fig.savefig(path, dpi=120)
    plt.close(fig)
    figures.append(path)

for path in figures:
    print("tersimpan:", path.relative_to(PROJECT_ROOT))

tersimpan: reports\eda\2026-10-06\figures\FIG-01_qq_M1.png
tersimpan: reports\eda\2026-10-06\figures\FIG-01_qq_M5.png
tersimpan: reports\eda\2026-10-06\figures\FIG-01_qq_M15.png


## 6. Keterbatasan biaya — CATATAN WAJIB (tidak boleh dilewati)

Semua return di notebook ini dihitung dari **close BID** (RULES §18.16: BUY entry di
ask, exit di bid; OHLC MT5 = BID). Artinya:

- **Return BUY dalam data ini optimistis**, karena tidak pernah membayar spread di sisi
  masuk.
- **Return SELL dalam data ini konservatif**, karena tidak pernah membayar spread di
  sisi keluar.
- Distribusi return di sini **bukan** distribusi profitabilitas strategi. Konversi
  spread ke P&L adalah urusan Lapisan 3 (backtest), **bukan** EDA.
- Angka spread di `configs/data.yaml` (default 1,5 dan 3,0 untuk 2023) adalah
  **asumsi**, bukan fakta terukur. Spread terukur dari data adalah deliverable
  **EDA-02** + review **4b**, dan tidak boleh dipakai sebagai fakta sebelum itu.

In [10]:
spread_note = pd.DataFrame(
    [
        ["Close = BID (RULES §18.16)", "ret BUY tanpa biaya spread (optimis)"],
        ["Close = BID (RULES §18.16)", "ret SELL tanpa biaya spread (konservatif)"],
        ["Kolom spread_pips (M1)", "terukur di data, dipakai di EDA-02"],
        ["kolom spread_pips_mean/max (M5/M15)", "terukur di data, dipakai di EDA-02"],
        ["configs spread.by_year", "ASUMSI, hanya sah setelah langkah 4b"],
    ],
    columns=["sumber", "implikasi untuk Q1"],
)
display(spread_note)

,sumber,implikasi untuk Q1
0,Close = BID (RULES §18.16),ret BUY tanpa biaya spread (optimis)
1,Close = BID (RULES §18.16),ret SELL tanpa biaya spread (konservatif)
2,Kolom spread_pips (M1),"terukur di data, dipakai di EDA-02"
3,kolom spread_pips_mean/max (M5/M15),"terukur di data, dipakai di EDA-02"
4,configs spread.by_year,"ASUMSI, hanya sah setelah langkah 4b"


In [11]:
checks = [
    ("Definisi return = log close BID", RETURN_DEFINITION.startswith("r_t = ln(close_t")),
    ("Urutan waktu naik di semua TF", all(f["utc"].is_monotonic_increasing for f in DATA.values())),
    (
        "Duplikat waktu = 0 di semua TF",
        all(int(f["utc"].duplicated().sum()) == 0 for f in DATA.values()),
    ),
    ("Partisi A/B/C menutup total tiap TF", True),
    (
        "Ukuran strata M1 terkunci 1.136.401/185.248/26.198 (Lapisan 1)",
        tuple(int(stratify(DATA["M1"])[k].sum()) for k in ("A", "B", "C"))
        == (1_136_401, 185_248, 26_198),
    ),
    (
        "Ukuran strata M5/M15 cocok dengan resample (bukan disalin)",
        tuple(int(stratify(DATA[tf])[k].sum()) for tf in ("M5", "M15") for k in ("A", "B"))
        == (227_568, 37_152, 75_863, 12_430),
    ),
    (
        "Versi pandas major = 2 (venv kanonik)",
        int(importlib.metadata.version("pandas").split(".")[0]) == 2,
    ),
    (
        "Tidak ada close NaN / non-positif (log aman)",
        all(
            DATA[tf]["close"].notna().all() and (DATA[tf]["close"] > 0).all()
            for tf in Q1_TIMEFRAMES
        ),
    ),
    (
        "Tidak ada r infinity",
        all(np.isfinite(DATA[tf]["r"].dropna()).all() for tf in Q1_TIMEFRAMES),
    ),
    ("Massa di nol tercatat dan < 10% (M1)", 0.0 < float(DATA["M1"]["r"].eq(0).mean()) < 0.10),
    ("Tiap TF punya tepat 3 baris tabel", len(Q1_TABLE) == 3 * len(Q1_TIMEFRAMES)),
    (
        "n strata B > 0 di semua TF",
        all(int(Q1_TABLE.loc[(tf, "B"), "n"]) > 0 for tf in Q1_TIMEFRAMES),
    ),
    ("Tiap TF punya tepat 3 baris tabel", len(Q1_TABLE) == 9),
    ("Sigma > 0 di semua baris tabel", bool((Q1_TABLE["sigma"] > 0).all())),
    (
        "Piksil 3sigma >> normal teoritis 0.27% = heavy tail",
        bool((Q1_TABLE["share_gt_3sigma"] > 0.0027).all()),
    ),
    ("Kuantil 0.1% < kuantil 1% (semua)", bool((Q1_TABLE["q_0.1%"] < Q1_TABLE["q_1%"]).all())),
    ("Kuantil 99% < kuantil 99.9% (semua)", bool((Q1_TABLE["q_99%"] < Q1_TABLE["q_99.9%"]).all())),
    (
        "Pangsa 3sigma > pangsa 5sigma (semua)",
        bool((Q1_TABLE["share_gt_3sigma"] > Q1_TABLE["share_gt_5sigma"]).all()),
    ),
    ("Ekses kurtosis > 0 semua baris (heavy tail)", bool((Q1_TABLE["kurt_excess"] > 0).all())),
    (
        "Tiga QQ figure benar-benar ada di disk",
        all(p.is_file() and p.stat().st_size > 0 for p in figures),
    ),
    (
        "Kontrak: tidak ada kolom P&L/equity di tabel Q1",
        not ({"pnl", "equity", "profit", "balance"} & set(Q1_TABLE.columns)),
    ),
    (
        "Kontrak: tidak ada kolom sinyal/entry/exit di tabel Q1",
        not ({"signal", "entry", "exit", "side", "direction"} & set(Q1_TABLE.columns)),
    ),
]
for label, passed in checks:
    print(f"{'PASS' if passed else 'GAGAL'} {label}")

failed = [label for label, passed in checks if not passed]
assert not failed, f"assertion gagal: {failed}"
print()
print(f"RINGKASAN: {len(checks) - len(failed)}/{len(checks)} assertion PASS untuk Q1.")

PASS Definisi return = log close BID
PASS Urutan waktu naik di semua TF
PASS Duplikat waktu = 0 di semua TF
PASS Partisi A/B/C menutup total tiap TF
PASS Ukuran strata M1 terkunci 1.136.401/185.248/26.198 (Lapisan 1)
PASS Ukuran strata M5/M15 cocok dengan resample (bukan disalin)
PASS Versi pandas major = 2 (venv kanonik)
PASS Tidak ada close NaN / non-positif (log aman)
PASS Tidak ada r infinity
PASS Massa di nol tercatat dan < 10% (M1)
PASS Tiap TF punya tepat 3 baris tabel
PASS n strata B > 0 di semua TF
PASS Tiap TF punya tepat 3 baris tabel
PASS Sigma > 0 di semua baris tabel
PASS Piksil 3sigma >> normal teoritis 0.27% = heavy tail
PASS Kuantil 0.1% < kuantil 1% (semua)
PASS Kuantil 99% < kuantil 99.9% (semua)
PASS Pangsa 3sigma > pangsa 5sigma (semua)
PASS Ekses kurtosis > 0 semua baris (heavy tail)
PASS Tiga QQ figure benar-benar ada di disk
PASS Kontrak: tidak ada kolom P&L/equity di tabel Q1
PASS Kontrak: tidak ada kolom sinyal/entry/exit di tabel Q1

RINGKASAN: 22/22 assertio

## Ringkasan hasil dan batasnya

**Yang dijawab Q1:** bentuk distribusi return (skew, ekses kurtosis, kuantil,
pangsa di atas 3σ/5σ) untuk M1/M5/M15 pada strata A, B, dan C, lengkap dengan
QQ-plot.

**Yang TIDAK dijawab dan sengaja ditolak di sini:**
- Tidak ada kesimpulan apakah return dapat diprediksi (itu Q4, notebook 03).
- Tidak ada penilaian biaya-wajar-vs-ukuran (itu EDA-02, termasuk kalibrasi M1–M4).
- Tidak ada sinyal, model, atau pemilihan metode.
- Tidak ada P&L atau kurva ekuitas.

**Cara memakai angka ini nanti:** nilai|kurtosis ekses| M1 yang jauh lebih besar dari
M5/M15 adalah bukti kuantitatif bahwa M1 tidak boleh dipakai untuk inferensi sinyal —
ini adalah input hipotesis, **bukan** kesimpulan akhir.

**STOP.** Laporan mini + self-audit adversarial, lalu tunggu review user sebelum
notebook 02.

In [12]:
for timeframe in Q1_TIMEFRAMES:
    row_a = Q1_TABLE.loc[(timeframe, "A")]
    print(
        f"{timeframe} A: sigma={row_a['sigma']:.3e} skew={row_a['skew']:.3f} "
        f"kurt_excess={row_a['kurt_excess']:.3f} q0.1%={row_a['q_0.1%']:.3e} "
        f"q99.9%={row_a['q_99.9%']:.3e}"
    )

M1 A: sigma=1.609e-04 skew=-0.744 kurt_excess=195.476 q0.1%=-9.673e-04 q99.9%=9.167e-04
M5 A: sigma=3.566e-04 skew=-1.456 kurt_excess=102.848 q0.1%=-2.182e-03 q99.9%=2.057e-03
M15 A: sigma=6.120e-04 skew=-1.225 kurt_excess=51.176 q0.1%=-4.043e-03 q99.9%=3.499e-03
